In [1]:
import dynamiqs as dq
dq.set_device('cpu')

import warnings
warnings.filterwarnings("ignore")

In [3]:
# Ec_t = 0.2
# def objective(params):
#     Ej_t = params[0]
#     qst = MyTransmon.create(
#         N = 10,
#         params = {"Ej": Ej_t, "Ec": Ec_t,"ng":0.0},
#         N_max_charge=20
#     )
#     return abs(qst.eig_systems["vals"][1] - qst.eig_systems["vals"][0] - 7.182920828753576)


# optimizer = optax.adam(learning_rate=0.1) 
# params = jnp.array([40.0])
# opt_state = optimizer.init(params)

# num_steps = 1000
# for step in range(num_steps):
#     val, grads = jax.value_and_grad(objective)(params)
#     if step %10 == 0:
#         clear_output()
#         print(f"iter: {step}, val={val:.4f} grads: {grads}, params: {params}")
#     if val < 1e-6:
#         clear_output()
#         print(f"iter: {step}, val={val:.4f} grads: {grads}, params: {params}")
#         break
#     updates, opt_state = optimizer.update(grads, opt_state)
#     params = optax.apply_updates(params, updates)

# print(f'Optimized params: {params}')

# Now our objective is: at the end of the pulse, there's little population for f1t1 and f2t1, but f0t1 as big as possible

In [13]:
solver = dq.solver.Tsit5(
                    rtol= 1e-06,
                    atol= 1e-06,
                    safety_factor= 0.9,
                    min_factor= 0.2,
                    max_factor = 5.0,
                    max_steps = int(1e4*1000),
                )

n_lvls_fluxonium = 20
n_lvls_transmon = 4


Ej_f = 2.7
Ec_f = 0.6
El_f = 0.13
qsf = qs.Fluxonium.create(
    n_lvls_fluxonium,
    {"Ej": Ej_f, "Ec": Ec_f, "El": El_f, "phi_ext": 0.0},
    N_pre_diag=100,
    use_linear = False
    )
g_tf = 0.2
Ec_t = 0.2

t_rise = 30

truncated_dim = 26 # will include 7,1
def truncate(data: jnp.array):
    return data[:truncated_dim,:truncated_dim]

tot_dim = truncated_dim


In [15]:

def objective(params):    
    pulse_length = params[0]
    amp_with_2pi = params[1]
    Ej_t = params[2]
    
    Ec_t = 0.2
    qst = MyTransmon.create(
        N = n_lvls_transmon,
        params = {"Ej": Ej_t, "Ec": Ec_t,"ng":0.0},
        N_max_charge=10
        )
    

    devices = [qsf, qst]
    f_indx = 0
    t_indx = 1
    Ns = [device.N for device in devices]
    fn = qs.promote(qsf.ops["n"], f_indx, Ns)
    tn = qs.promote(qst.ops['n'], t_indx, Ns)

    g_tf = 0.2
    system = qs.System.create(devices, couplings=[
        g_tf *  fn @ tn
        ])
    system.params["g_tf"] = g_tf
    system_evals_in_product_indices, system_evecs_in_product_indices = system.calculate_eig_linear()
    system_evals_sorted, system_evecs_sorted, product_indices_sorted_by_eval = calculate_eig(Ns, system.get_H())
    driven_op = transform_op_into_dressed_basis_jax(tn, system_evecs_sorted.T)

    w_d = system_evals_in_product_indices[0,1] - system_evals_in_product_indices[0,0]
    pulse_shape_args={
        'w_d': w_d,
        'amp': amp_with_2pi/(2*jnp.pi),
        't_rise': t_rise,
        't_square': pulse_length - t_rise
    }      

    t_tot = pulse_length + t_rise
    tlist = jnp.array([0,t_tot])

    def _H(t):
        _H =  2 * jnp.pi *truncate(jnp.diag(system_evals_sorted))
        _H += truncate(driven_op) * square_pulse_with_rise_fall(t, pulse_shape_args)
        return _H 
    H =  dq.timecallable(_H)
    
    psi0_list = [truncate(dq.basis(tot_dim,find_closest_dressed_index(l*qst.N, product_indices_sorted_by_eval)))
                        for l in [0,1,2]] #00,10,20
    result = dq.sesolve(
                H = H,
                psi0 = psi0_list,
                tsave = tlist,
                solver = solver
                )
        
    f0_e = dq.expect(dq.basis_dm(tot_dim, find_closest_dressed_index(0*qst.N+1, product_indices_sorted_by_eval)),
                    result.states[0][-1]).real
    
    f1_e = dq.expect(dq.basis_dm(tot_dim, find_closest_dressed_index(1*qst.N+1, product_indices_sorted_by_eval)),
                    result.states[1][-1]).real
    
    f2_e = dq.expect(dq.basis_dm(tot_dim, find_closest_dressed_index(2*qst.N+1, product_indices_sorted_by_eval)),
                    result.states[2][-1]).real
    
    return 1 - f0_e + f1_e + f2_e

func = jax.value_and_grad(objective)



In [16]:
params =  jnp.array([
                     250.0,
                     0.011026707187734986,
                     34.12111946
            ])



In [17]:


optimizer = optax.nadam(learning_rate=jnp.array([2,0.004,0.1])) 
opt_state = optimizer.init(params)

num_steps = 1000
for step in range(num_steps):
    val, grads = func(params)
    # clear_output()
    print(f"iter: {step}, val={val:.4f} grads: {grads}, params: {params}")
    if val < 1e-4:
        break
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

print(f'Optimized params: {params}')

iter: 0, val=0.1594 grads: [8.51340728e-03 1.20477081e+02 1.16970881e+00], params: [2.50000000e+02 1.10267072e-02 3.41211195e+01]
iter: 1, val=0.4447 grads: [-4.42066212e-03 -1.50368429e+02 -3.51831866e-01], params: [2.47052635e+02 5.13197035e-03 3.39737510e+01]
iter: 2, val=0.1584 grads: [-4.43865818e-03 -4.21552907e+01 -1.05772829e+00], params: [2.47421306e+02 7.86423394e-03 3.39682390e+01]
iter: 3, val=0.0647 grads: [-3.03264856e-03  1.32127711e+01 -1.44045685e+00], params: [2.48088740e+02 9.14534881e-03 3.40220567e+01]
iter: 4, val=0.0204 grads: [2.04331943e-03 2.78202637e+01 3.81621217e-01], params: [2.48761689e+02 9.51208166e-03 3.40977956e+01]
iter: 5, val=0.0269 grads: [2.64525366e-03 2.34869655e+01 7.71196209e-01], params: [2.48709484e+02 9.42285806e-03 3.41129483e+01]
iter: 6, val=0.0158 grads: [1.37804992e-03 6.38746560e+00 4.61505131e-01], params: [2.48401063e+02 9.17006243e-03 3.41022306e+01]
iter: 7, val=0.0120 grads: [ 2.03048819e-04 -2.36540339e+00  9.74812195e-02], par